# Sprint 4: Platform Routing Evaluation

Validates `models/router.py` against the locked frontend JSON contract.

**Contents:**
1. Schema validation function
2. Five stress-test items with full JSON output
3. Ranking accuracy evaluation vs. EDA ground truth

**Key caveats:**
- Depop `final_sale_price` is a listed-price proxy, not a sold price
- Sell-through estimates for eBay/Depop are documented assumptions (see `router.py` constants)
- Pricing model MedAPE is 50% — router value is in *relative* platform ranking, not absolute price accuracy

In [1]:
import json
import sys
sys.path.insert(0, "..")

from models.router import recommend_listing

## Section 1: Schema Validation

In [2]:
EXPECTED_ITEM_KEYS = {"category", "brand", "size", "condition", "color", "estimated_retail"}

EXPECTED_REC_KEYS = {
    "platform", "rank", "fit_score", "price_tiers", "platform_fee_pct",
    "estimated_shipping", "net_profit", "estimated_time_minutes",
    "effective_hourly_rate", "sell_probability_30d", "estimated_days_to_sale",
    "reasoning",
}

EXPECTED_TIER_KEYS = {"fast_sale", "balanced", "max_revenue"}

EXPECTED_WORTH_IT_KEYS = {
    "verdict", "best_net_profit", "best_platform",
    "effective_hourly_rate", "explanation",
}


def validate_response(response: dict) -> list[str]:
    """Return list of schema violations (empty = valid)."""
    errors = []

    # Top-level keys
    top_keys = set(response.keys())
    expected_top = {"item", "recommendations", "worth_it"}
    if top_keys != expected_top:
        errors.append(f"Top-level keys: got {top_keys}, expected {expected_top}")

    # Item block
    item = response.get("item", {})
    if set(item.keys()) != EXPECTED_ITEM_KEYS:
        errors.append(f"Item keys: got {set(item.keys())}, expected {EXPECTED_ITEM_KEYS}")

    # Recommendations
    recs = response.get("recommendations", [])
    if len(recs) != 3:
        errors.append(f"Expected 3 recommendations, got {len(recs)}")

    for i, rec in enumerate(recs):
        if set(rec.keys()) != EXPECTED_REC_KEYS:
            errors.append(f"Rec[{i}] keys: got {set(rec.keys())}, expected {EXPECTED_REC_KEYS}")
        if rec.get("rank") != i + 1:
            errors.append(f"Rec[{i}] rank should be {i + 1}, got {rec.get('rank')}")
        for tier_field in ["price_tiers", "net_profit"]:
            tiers = rec.get(tier_field, {})
            if set(tiers.keys()) != EXPECTED_TIER_KEYS:
                errors.append(f"Rec[{i}].{tier_field} keys: got {set(tiers.keys())}")
        if not isinstance(rec.get("reasoning"), str) or len(rec["reasoning"]) < 10:
            errors.append(f"Rec[{i}] reasoning too short or wrong type")

    # Worth it
    worth = response.get("worth_it", {})
    if set(worth.keys()) != EXPECTED_WORTH_IT_KEYS:
        errors.append(f"worth_it keys: got {set(worth.keys())}, expected {EXPECTED_WORTH_IT_KEYS}")
    if worth.get("verdict") not in (True, False, "marginal"):
        errors.append(f"worth_it.verdict unexpected value: {worth.get('verdict')}")

    return errors


print("Schema validator ready.")

Schema validator ready.


## Section 2: Stress-Test Items

In [3]:
TEST_ITEMS = [
    {
        "name": "Clear worth-it (Levi's denim jacket)",
        "item": {
            "category": "denim jacket",
            "brand": "Levi's",
            "condition": "Like New",
            "size": "M",
            "color": "blue",
            "estimated_retail": 89.99,
        },
    },
    {
        "name": "Likely not-worth-it (H&M midi dress)",
        "item": {
            "category": "midi dress",
            "brand": "H&M",
            "condition": "Good",
            "size": "S",
            "color": "black",
            "estimated_retail": 24.99,
        },
    },
    {
        "name": "Platform-divergent (Nike sneakers)",
        "item": {
            "category": "sneakers",
            "brand": "Nike",
            "condition": "New",
            "size": "10",
            "color": "white",
            "estimated_retail": 120.00,
        },
    },
    {
        "name": "Luxury (Louis Vuitton handbag)",
        "item": {
            "category": "handbag",
            "brand": "Louis Vuitton",
            "condition": "Like New",
            "size": "OS",
            "color": "brown",
            "estimated_retail": 2000.00,
        },
    },
    {
        "name": "No-brand (Unknown blazer)",
        "item": {
            "category": "blazer",
            "brand": "Unknown",
            "condition": "Good",
            "size": "L",
            "color": "gray",
            "estimated_retail": 40.00,
        },
    },
]

print(f"{len(TEST_ITEMS)} test items defined.")

5 test items defined.


In [4]:
results = []
all_valid = True

for test in TEST_ITEMS:
    print(f"{'=' * 70}")
    print(f"TEST: {test['name']}")
    print(f"{'=' * 70}")

    result = recommend_listing(test["item"])
    results.append(result)

    # Print full JSON
    print(json.dumps(result, indent=2))

    # Validate schema
    errors = validate_response(result)
    if errors:
        all_valid = False
        print(f"\n\u274c SCHEMA ERRORS:")
        for e in errors:
            print(f"  - {e}")
    else:
        print(f"\n\u2705 Schema valid")

    # Commentary
    best = result["recommendations"][0]
    verdict = result["worth_it"]["verdict"]
    print(f"\nRank 1: {best['platform']} (fit_score {best['fit_score']})")
    print(f"Verdict: {verdict} (${result['worth_it']['effective_hourly_rate']:.0f}/hr)")
    print()

print(f"{'=' * 70}")
if all_valid:
    print("\u2705 ALL 5 ITEMS PASS SCHEMA VALIDATION")
else:
    print("\u274c SOME ITEMS FAILED SCHEMA VALIDATION")

TEST: Clear worth-it (Levi's denim jacket)
{
  "item": {
    "category": "denim jacket",
    "brand": "Levi's",
    "size": "M",
    "condition": "Like New",
    "color": "blue",
    "estimated_retail": 89.99
  },
  "recommendations": [
    {
      "platform": "eBay",
      "rank": 1,
      "fit_score": 10.0,
      "price_tiers": {
        "fast_sale": 20,
        "balanced": 47,
        "max_revenue": 85
      },
      "platform_fee_pct": 0.13,
      "estimated_shipping": 5.99,
      "net_profit": {
        "fast_sale": 11.41,
        "balanced": 34.9,
        "max_revenue": 67.96
      },
      "estimated_time_minutes": 35,
      "effective_hourly_rate": 59.83,
      "sell_probability_30d": 1.0,
      "estimated_days_to_sale": 29,
      "reasoning": "Denim Jackets nets $10 more than Poshmark and estimated 10 days faster on eBay after 13% fee"
    },
    {
      "platform": "Poshmark",
      "rank": 2,
      "fit_score": 4.8,
      "price_tiers": {
        "fast_sale": 27,
        "ba

## Section 3: Ranking Accuracy vs. EDA Ground Truth

Compare the router's rank-1 platform against the EDA's "best platform by median net profit" for a generic item in each category (Unknown brand, Good condition).

**EDA ground truth** (from `01_eda.ipynb` Section 5a):

| Category | Best Platform (EDA) |
|----------|--------------------|
| blazer | eBay |
| crossbody bag | Poshmark |
| denim jacket | eBay |
| handbag | eBay |
| leather jacket | eBay |
| midi dress | Poshmark |
| sneakers | eBay |
| vintage t-shirt | eBay |

In [5]:
EDA_GROUND_TRUTH = {
    "blazer": "eBay",
    "crossbody bag": "Poshmark",
    "denim jacket": "eBay",
    "handbag": "eBay",
    "leather jacket": "eBay",
    "midi dress": "Poshmark",
    "sneakers": "eBay",
    "vintage t-shirt": "eBay",
}

# Test with a generic item per category (Unknown brand, Good condition)
# to isolate category-level routing from brand-specific effects
matches = 0
total = len(EDA_GROUND_TRUTH)

print(f"{'Category':<20} {'EDA Best':<12} {'Router Best':<12} {'Match'}")
print("-" * 55)

for category, eda_best in sorted(EDA_GROUND_TRUTH.items()):
    result = recommend_listing({
        "category": category,
        "brand": "Unknown",
        "condition": "Good",
    })
    router_best = result["recommendations"][0]["platform"]
    match = router_best == eda_best
    if match:
        matches += 1
    icon = "\u2705" if match else "\u274c"
    print(f"{category:<20} {eda_best:<12} {router_best:<12} {icon}")

print(f"\nRouting accuracy: {matches}/{total} ({matches/total:.0%})")
if matches < total:
    print("\nDisagreements are not necessarily wrong — the router uses a trained model,")
    print("while EDA ground truth is based on raw median net profit across all brands.")

Category             EDA Best     Router Best  Match
-------------------------------------------------------
blazer               eBay         Poshmark     ❌
crossbody bag        Poshmark     Poshmark     ✅
denim jacket         eBay         eBay         ✅
handbag              eBay         eBay         ✅
leather jacket       eBay         eBay         ✅
midi dress           Poshmark     Poshmark     ✅
sneakers             eBay         eBay         ✅
vintage t-shirt      eBay         eBay         ✅

Routing accuracy: 7/8 (88%)

Disagreements are not necessarily wrong — the router uses a trained model,
while EDA ground truth is based on raw median net profit across all brands.


## Section 4: Branded Item Routing Comparison

Test how brand affects routing: same category, different brands.

In [6]:
brand_tests = [
    ("handbag", "Coach"),
    ("handbag", "Louis Vuitton"),
    ("handbag", "Unknown"),
    ("sneakers", "Nike"),
    ("sneakers", "Adidas"),
    ("sneakers", "Unknown"),
]

print(f"{'Category':<15} {'Brand':<18} {'#1 Platform':<12} {'Balanced $':<12} {'Hourly $':<10} {'Verdict'}")
print("-" * 80)

for cat, brand in brand_tests:
    r = recommend_listing({"category": cat, "brand": brand, "condition": "Like New"})
    best = r["recommendations"][0]
    print(
        f"{cat:<15} {brand:<18} {best['platform']:<12} "
        f"${best['price_tiers']['balanced']:<10} "
        f"${r['worth_it']['effective_hourly_rate']:<9.0f} "
        f"{r['worth_it']['verdict']}"
    )

Category        Brand              #1 Platform  Balanced $   Hourly $   Verdict
--------------------------------------------------------------------------------
handbag         Coach              eBay         $107        $149       True
handbag         Louis Vuitton      eBay         $210        $303       True
handbag         Unknown            eBay         $31         $36        True
sneakers        Nike               eBay         $47         $60        True
sneakers        Adidas             eBay         $55         $72        True
sneakers        Unknown            eBay         $55         $72        True


## Section 5: Summary

Key takeaways from routing evaluation.

In [7]:
print("=== Sprint 4 Routing Evaluation Summary ===\n")

print("1. [Schema Compliance]")
print(f"   All {len(TEST_ITEMS)} test items pass schema validation: {'YES' if all_valid else 'NO'}")
print()

print("2. [Ranking Accuracy]")
print(f"   Router agrees with EDA ground truth: {matches}/{total} categories")
print()

print("3. [Worth It Verdicts]")
for test, result in zip(TEST_ITEMS, results):
    v = result["worth_it"]
    print(f"   {test['name']}: {v['verdict']} (${v['effective_hourly_rate']:.0f}/hr)")
print()

print("4. [Key Assumptions]")
print("   - eBay velocity = 0.75x Poshmark (assumption, not data)")
print("   - Depop velocity = 1.1x Poshmark (assumption, not data)")
print("   - Depop predictions are listed-price proxies")
print("   - Pricing model MedAPE = 50% — router value is relative ranking")
print()

print("5. [Depop Caveat]")
print("   All Depop price predictions are based on listed prices, not sold prices.")
print("   Depop does not expose sold-price history publicly.")

=== Sprint 4 Routing Evaluation Summary ===

1. [Schema Compliance]
   All 5 test items pass schema validation: YES

2. [Ranking Accuracy]
   Router agrees with EDA ground truth: 7/8 categories

3. [Worth It Verdicts]
   Clear worth-it (Levi's denim jacket): True ($60/hr)
   Likely not-worth-it (H&M midi dress): marginal ($20/hr)
   Platform-divergent (Nike sneakers): True ($87/hr)
   Luxury (Louis Vuitton handbag): True ($473/hr)
   No-brand (Unknown blazer): marginal ($13/hr)

4. [Key Assumptions]
   - eBay velocity = 0.75x Poshmark (assumption, not data)
   - Depop velocity = 1.1x Poshmark (assumption, not data)
   - Depop predictions are listed-price proxies
   - Pricing model MedAPE = 50% — router value is relative ranking

5. [Depop Caveat]
   All Depop price predictions are based on listed prices, not sold prices.
   Depop does not expose sold-price history publicly.
